# Pending comments: ECOD baseline and white-box gradient attack
Runtime > Run all. Allow Google Drive access when asked. Part 1 compares the ensemble detector with ECOD (a recent empirical-CDF outlier detector) under the same protocol. Part 2 attacks original malware flows with a gradient-based attack on a surrogate neural network (no queries to the target models) and reports how many of the resulting flows evade the IDS and how many are still flagged by the detector.

In [ ]:
import os, time, pickle, warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")

try:
    from google.colab import drive
except ImportError:
    drive = None                      # not running in Colab
if drive is not None:
    drive.mount("/content/drive")     # errors are shown, not hidden
    assert os.path.exists("/content/drive/MyDrive"), "Google Drive did not mount - re-run and click Allow"

# ---------------- CONFIG (edit here) ----------------
BASE = os.environ.get("DRELAB_BASE", "/content/drive/MyDrive/ColabNotebooks/DReLAB")
OUT = os.environ.get("REV_OUT", "/content/drive/MyDrive/ColabNotebooks/revision_results")
FAMILIES = {"CTU": ["Virut", "Neris", "Rbot", "Menti", "Murlo"],
            "BOTNET": ["Virut", "Neris", "Rbot", "Menti", "Murlo"],
            "CICIDS2018": ["Zeus_Ares"]}
ATTACKS = [("RF", "DDQN"), ("RF", "SARSA"), ("WnD", "DDQN"), ("WnD", "SARSA")]
SEEDS = list(range(int(os.environ.get("N_SEEDS", "5"))))
N_CHUNKS = int(os.environ.get("N_CHUNKS", "20"))       # position blocks
BUFFER = 0.02                                          # drop rows within 2% of a chunk edge
MAX_BENIGN = int(os.environ.get("MAX_BENIGN", "150000"))        # systematic (ordered) subsample of benign
MAX_TRAIN_PER_CLASS = int(os.environ.get("MAX_TRAIN", "40000"))
MAX_EVAL_PER_SET = int(os.environ.get("MAX_EVAL", "30000"))     # cap per val/test set (speed)
FAR_TARGET_PCT = 95                                    # threshold = 95th percentile of clean validation scores
BITS = 8                                               # quantization bits for the cleaned-up squeezer
DROP = {"Unnamed: 0", "Label", "label", "n_actions"}
os.makedirs(OUT, exist_ok=True)


In [ ]:
# ---------------- DATA ----------------
def with_pos(df):
    df = df.copy()
    df["_pos"] = (np.arange(len(df)) + 0.5) / len(df)
    return df

def load_dataset(ds):
    d = f"{BASE}/{ds}"
    benign = pd.read_csv(f"{d}/benign.csv")
    if len(benign) > MAX_BENIGN:                       # ordered subsample, keeps temporal structure
        benign = benign.iloc[np.linspace(0, len(benign) - 1, MAX_BENIGN).astype(int)]
    fams = FAMILIES[ds]
    orig = [with_pos(pd.read_csv(f"{d}/malicious/{f}.csv")) for f in fams]
    adv = {(o, a): [with_pos(pd.read_csv(f"{d}/{o}/{a}/{f}.csv")) for f in fams] for o, a in ATTACKS}
    benign = with_pos(benign)
    ref = [benign] + orig + [x for v in adv.values() for x in v]
    feats = [c for c in benign.columns
             if c not in DROP and c != "_pos"
             and all(c in r.columns for r in ref)
             and all(pd.api.types.is_numeric_dtype(r[c]) for r in ref)]
    def pool(frames):
        df = pd.concat(frames, ignore_index=True)
        X = df[feats].replace([np.inf, -np.inf], np.nan)
        X = X.fillna(X.median()).values.astype(np.float32)
        return X, df["_pos"].values
    data = {"benign": pool([benign]), "orig": pool(orig)}
    for k, v in adv.items():
        data[k] = pool(v)
    return data, feats

def split_roles(seed):
    rng = np.random.RandomState(seed)
    perm = rng.permutation(N_CHUNKS)
    n_tr, n_va = int(N_CHUNKS * 0.5), int(N_CHUNKS * 0.2)
    role = np.empty(N_CHUNKS, dtype=object)
    role[perm[:n_tr]] = "train"
    role[perm[n_tr:n_tr + n_va]] = "val"
    role[perm[n_tr + n_va:]] = "test"
    return role

def take(pool, role, split, rng, cap):
    X, pos = pool
    ch = np.minimum((pos * N_CHUNKS).astype(int), N_CHUNKS - 1)
    frac = pos * N_CHUNKS - ch
    m = (role[ch] == split) & (frac > BUFFER) & (frac < 1 - BUFFER)
    Xs = X[m]
    if len(Xs) > cap:
        Xs = Xs[rng.choice(len(Xs), cap, replace=False)]
    return Xs


In [ ]:
# ---------------- SQUEEZERS ----------------
def make_quant(Xtr, bits=BITS, q=(0.5, 99.5)):
    """True k-bit quantization: clip to the training [q_lo, q_hi] range, then 2^k levels per feature."""
    lo = np.percentile(Xtr, q[0], axis=0); hi = np.percentile(Xtr, q[1], axis=0)
    span = np.maximum(hi - lo, 1e-8); L = 2 ** bits - 1
    def sq(X):
        Z = np.clip((X - lo) / span, 0, 1)
        return (np.round(Z * L) / L * span + lo).astype(np.float32)
    return sq

def paper_sq(X):
    """The original paper code: round(x*256)/256, then a trailing median over 3 adjacent feature columns."""
    X1 = np.round(X * 256) / 256
    s1 = np.full_like(X1, np.nan); s1[:, 1:] = X1[:, :-1]
    s2 = np.full_like(X1, np.nan); s2[:, 2:] = X1[:, :-2]
    return np.nanmedian(np.stack([X1, s1, s2]), axis=0).astype(np.float32)

# ---------------- SCORES ----------------
def conf(p): return np.maximum(p, 1 - p)

def base_probs(X, rf, xgb):
    return rf.predict_proba(X)[:, 1], xgb.predict_proba(X)[:, 1]

def independent_scores(X, rf, xgb, rng):
    """Detectors that do not depend on the squeezer."""
    p_rf, p_xg = base_probs(X, rf, xgb)
    pn = rf.predict_proba(X + rng.normal(0, 0.01, X.shape).astype(np.float32))[:, 1]
    return {"Noise sensitivity (RF)": np.abs(p_rf - pn),
            "Ensemble disagreement only": np.abs(p_rf - p_xg)}

def squeeze_scores(X, rf, xgb, sq):
    p_rf, p_xg = base_probs(X, rf, xgb)
    Xs = sq(X)
    s_rf, s_xg = base_probs(Xs, rf, xgb)
    return {"Feature squeezing (RF)": np.abs(p_rf - s_rf),
            "Confidence drop (RF)": np.abs(conf(p_rf) - conf(s_rf)),
            "Squeeze + ensemble, Eq.11 variance": np.var(np.vstack([s_rf, s_xg]), axis=0),
            "Squeeze + ensemble, Eq.24 sum": np.abs(p_rf - s_rf) + np.abs(p_xg - s_xg)}

PROPOSED = ["Squeeze + ensemble, Eq.11 variance", "Squeeze + ensemble, Eq.24 sum"]
BASELINES = ["Noise sensitivity (RF)", "Feature squeezing (RF)", "Confidence drop (RF)"]

# ---------------- METRICS ----------------
def metrics(neg_val, neg_te, pos_te):
    thr = np.percentile(neg_val, FAR_TARGET_PCT)
    tp = int((pos_te > thr).sum()); fn = len(pos_te) - tp
    fp = int((neg_te > thr).sum()); tn = len(neg_te) - fp
    tpr, fpr = tp / len(pos_te), fp / len(neg_te)
    prec = tpr / (tpr + fpr + 1e-12)                  # precision at balanced prevalence
    f1 = 2 * prec * tpr / (prec + tpr + 1e-12)
    y = np.r_[np.zeros(len(neg_te)), np.ones(len(pos_te))]; s = np.r_[neg_te, pos_te]
    return dict(thr=thr, TPR=tpr, FPR=fpr, Precision_bal=prec, F1_bal=f1,
                AUROC=roc_auc_score(y, s), AUPRC=average_precision_score(y, s),
                TP=tp, FN=fn, FP=fp, TN=tn)


In [ ]:
# ---- PENDING COMMENTS: (1) ECOD, a recent tabular outlier detector (R1-2); (2) white-box gradient attack via a surrogate network (R1-4) ----
from sklearn.neural_network import MLPClassifier
from scipy.stats import skew
from sklearn.metrics import roc_auc_score
N_ATTACK, FMAX, WB_STEPS, WB_ALPHA, WB_EPS, SURR_CAP = 2000, 10.0, 30, 0.15, 0.05, 30000
NEED = ["Dur", "InBytes", "OutBytes", "TotPkts", "TotBytes", "BytesPerPkt", "BytesPerSec", "PktsPerSec", "RatioOutIn"]

def split_sets(D, seed):          # identical sampling to the earlier runs (all features)
    role = split_roles(seed); rng = np.random.RandomState(seed)
    g = lambda k, sp, cap: take(D[k], role, sp, rng, cap)
    Xb, Xo = g("benign", "train", 10**9), g("orig", "train", 10**9)
    n = min(len(Xb), len(Xo), MAX_TRAIN_PER_CLASS)
    Xb, Xo = Xb[rng.choice(len(Xb), n, replace=False)], Xo[rng.choice(len(Xo), n, replace=False)]
    return dict(train=(np.vstack([Xb, Xo]), np.r_[np.zeros(n), np.ones(n)]),
                val=np.vstack([g("benign", "val", MAX_EVAL_PER_SET), g("orig", "val", MAX_EVAL_PER_SET)]),
                neg=np.vstack([g("benign", "test", MAX_EVAL_PER_SET), g("orig", "test", MAX_EVAL_PER_SET)]),
                adv=np.vstack([g(k, "test", MAX_EVAL_PER_SET) for k in ATTACKS]))

class ECOD:
    """Empirical-CDF outlier detector (Li et al., 2022): per-feature tail probabilities, aggregated as max(left, right, skew-chosen)."""
    def fit(self, Z):
        self.s = np.sort(Z, axis=0); self.n = len(Z); self.skew = np.nan_to_num(skew(Z, axis=0)); return self
    def score(self, X):
        n = self.n; oL = np.empty(X.shape); oR = np.empty(X.shape)
        for j in range(X.shape[1]):
            oL[:, j] = -np.log((np.searchsorted(self.s[:, j], X[:, j], "right") + 1) / (n + 1))
            oR[:, j] = -np.log((n - np.searchsorted(self.s[:, j], X[:, j], "left") + 1) / (n + 1))
        oS = np.where(self.skew < 0, oL, oR)
        return np.maximum.reduce([oL.sum(1), oR.sum(1), oS.sum(1)])

def perturb(X0, F, ix, rnd=True):
    """Scale duration, bytes in, bytes out and packet count by factors >= 1; keep derived features consistent."""
    X = X0.copy(); r = np.round if rnd else (lambda a: a)
    dur0, in0, out0, pk0, tot0 = (X0[:, ix[c]] for c in ("Dur", "InBytes", "OutBytes", "TotPkts", "TotBytes"))
    dur = dur0 * F[:, 0]; inb = r(in0 * F[:, 1]); outb = r(out0 * F[:, 2]); pk = np.maximum(r(pk0 * F[:, 3]), 1)
    tot = tot0 + (inb - in0) + (outb - out0)
    ratio = lambda a, b: np.where(b > 0, a / np.maximum(b, 1e-12), 1.0)
    kd, kt, kp, ko, ki = ratio(dur, dur0), ratio(tot, tot0), ratio(pk, pk0), ratio(outb, out0), ratio(inb, in0)
    X[:, ix["Dur"]] = dur; X[:, ix["InBytes"]] = inb; X[:, ix["OutBytes"]] = outb; X[:, ix["TotPkts"]] = pk; X[:, ix["TotBytes"]] = tot
    X[:, ix["BytesPerPkt"]] = X0[:, ix["BytesPerPkt"]] * kt / np.maximum(kp, 1e-12)
    X[:, ix["BytesPerSec"]] = X0[:, ix["BytesPerSec"]] * kt / np.maximum(kd, 1e-12)
    X[:, ix["PktsPerSec"]] = X0[:, ix["PktsPerSec"]] * kp / np.maximum(kd, 1e-12)
    X[:, ix["RatioOutIn"]] = X0[:, ix["RatioOutIn"]] * ko / np.maximum(ki, 1e-12)
    return X

def mlp_logit_grad(mlp, Z):
    """Exact gradient of the surrogate's malware logit with respect to its (standardised) input."""
    a, pres = Z, []
    for W, b in zip(mlp.coefs_[:-1], mlp.intercepts_[:-1]):
        pre = a @ W + b; pres.append(pre); a = np.maximum(pre, 0)
    g = np.repeat(mlp.coefs_[-1].T, len(Z), axis=0)
    for W, pre in zip(reversed(mlp.coefs_[:-1]), reversed(pres)): g = (g * (pre > 0)) @ W.T
    return g

def whitebox_pgd(X0, ix, scaler, mlp):
    """Signed-gradient descent on the surrogate's logit in log-factor space (factors in [1, FMAX]); zero queries to the target models."""
    X0 = X0.astype(np.float64); n = len(X0); L = np.zeros((n, 4)); LMAX = np.log(FMAX)
    for _ in range(WB_STEPS):
        X = perturb(X0, np.exp(L), ix, rnd=False)
        gX = mlp_logit_grad(mlp, ((X - scaler.mean_) / scaler.scale_).astype(np.float32)) / scaler.scale_
        grad = np.zeros((n, 4))
        for k in range(4):
            Lk = L.copy(); Lk[:, k] += WB_EPS
            grad[:, k] = ((perturb(X0, np.exp(Lk), ix, rnd=False) - X) * gX).sum(1) / WB_EPS
        L = np.clip(L - WB_ALPHA * np.sign(grad), 0, LMAX)
    return perturb(X0, np.exp(L), ix, rnd=True)

rows, wb_rows = [], []
for ds in FAMILIES:
    print("loading", ds)
    D, feats = load_dataset(ds); ix = {c: feats.index(c) for c in NEED}
    for seed in SEEDS:
        t0 = time.time()
        S = split_sets(D, seed); Xtr, ytr = S["train"]
        sc = StandardScaler().fit(Xtr); T = lambda X, sc=sc: sc.transform(X).astype(np.float32); Z = T(Xtr)
        val, neg, adv = T(S["val"]), T(S["neg"]), T(S["adv"])
        rf = RandomForestClassifier(150, max_depth=20, n_jobs=-1, random_state=seed).fit(Z, ytr)
        xg = XGBClassifier(n_estimators=150, max_depth=8, learning_rate=0.1, eval_metric="logloss", random_state=seed).fit(Z, ytr)
        ens = lambda Zx: np.abs(rf.predict_proba(Zx)[:, 1] - xg.predict_proba(Zx)[:, 1])
        ecod = ECOD().fit(Z)
        for name, f in {"Ensemble disagreement only": ens, "ECOD (empirical-CDF outlier score)": ecod.score}.items():
            m = metrics(f(val), f(neg), f(adv))
            rows.append(dict(dataset=ds, seed=seed, method=name, TPR=m["TPR"], FPR=m["FPR"], AUROC=m["AUROC"], AUPRC=m["AUPRC"]))
        # ---- white-box attack on a surrogate network, transferred to RF and XGBoost
        idx = np.random.RandomState(seed).choice(len(Z), min(len(Z), SURR_CAP), replace=False)
        mlp = MLPClassifier(hidden_layer_sizes=(64, 32), activation="relu", early_stopping=True, max_iter=60, random_state=seed).fit(Z[idx], ytr[idx])
        tau = np.percentile(ens(val), FAR_TARGET_PCT); s_neg = ens(neg)
        X_all = take(D["orig"], split_roles(seed), "test", np.random.RandomState(1000 + seed), 5 * N_ATTACK)
        X0 = X_all[rf.predict_proba(T(X_all))[:, 1] >= 0.5][:N_ATTACK]
        Xa = whitebox_pgd(X0, ix, sc, mlp).astype(np.float32); Za = T(Xa)
        pr, px = rf.predict_proba(Za)[:, 1], xg.predict_proba(Za)[:, 1]; Dd = np.abs(pr - px)
        ev_s = mlp.predict_proba(Za)[:, 1] < 0.5; ev_rf = pr < 0.5; ev_both = ev_rf & (px < 0.5); fl = Dd > tau
        wb_rows.append(dict(dataset=ds, seed=seed, n=len(X0), evade_surrogate=ev_s.mean(), evade_RF=ev_rf.mean(), evade_both=ev_both.mean(),
                            flagged_all=fl.mean(), flagged_among_RF_evaders=(fl[ev_rf].mean() if ev_rf.any() else np.nan),
                            undetected_RF_evasion=(ev_rf & ~fl).mean(), undetected_both_evasion=(ev_both & ~fl).mean(),
                            AUROC=roc_auc_score(np.r_[np.zeros(len(s_neg)), np.ones(len(Dd))], np.r_[s_neg, Dd])))
        print(f"  {ds} seed {seed} done in {time.time() - t0:.0f}s ({len(X0)} flows attacked)")
    del D

res = pd.DataFrame(rows); res.to_csv(f"{OUT}/ecod_results.csv", index=False)
wb = pd.DataFrame(wb_rows); wb.to_csv(f"{OUT}/whitebox_attack_results.csv", index=False)
pm = lambda x: f"{np.mean(x):.3f}+/-{np.std(x, ddof=1) if len(x) > 1 else 0:.3f}"
pd.set_option("display.width", 250, "display.max_colwidth", 45)
print("\n===== ECOD vs ensemble disagreement (pooled attacks, 95th-percentile threshold) =====")
print(res.groupby(["dataset", "method"]).agg(TPR=("TPR", pm), FPR=("FPR", pm), AUROC=("AUROC", pm), AUPRC=("AUPRC", pm)).reset_index().to_string(index=False))
print("\n===== Paired t-test on AUROC: ensemble vs ECOD =====")
for ds in FAMILIES:
    a = res[(res.dataset == ds) & (res.method == "Ensemble disagreement only")].sort_values("seed").AUROC.values
    b = res[(res.dataset == ds) & (res.method.str.startswith("ECOD"))].sort_values("seed").AUROC.values
    t, p = stats.ttest_rel(a, b); print(f"{ds:10s} ensemble {a.mean():.3f} vs ECOD {b.mean():.3f}  diff={a.mean() - b.mean():+.3f}  p={p:.4f}  ensemble wins {int((a > b).sum())}/{len(a)}")
print("\n===== White-box gradient attack on a surrogate network, transferred (0 queries to the target models) =====")
cols = ["evade_surrogate", "evade_RF", "evade_both", "flagged_all", "flagged_among_RF_evaders", "undetected_RF_evasion", "undetected_both_evasion", "AUROC"]
print(wb.groupby("dataset")[cols].agg(pm).reset_index().to_string(index=False))
print("(Seeds share the same underlying flows, so p-values are indicative, not formal.)")
